# Benchmark controls and summary tables

This notebook runs two validity controls for the rare-cell downsampling benchmark and then
generates aggregated summary tables from the raw results CSV.

**Abundant-cell control** — runs the same benchmark on the most-abundant non-target cell type.
Recovery should stay high at all fractions because those cells are never artificially depleted.

**Random-label control** — randomly permutes the label column before evaluation at the
smallest retention fraction.  Recovery should collapse to near-chance.

**Summary tables** — aggregates the raw benchmark CSV into seed-level mean/std tables
and identifies the best representation per fraction.

Mirrors:
- `scripts/run_controls.py`
- `scripts/generate_summary_tables.py`

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
assert (PROJECT_ROOT / "src" / "rarecell").exists(), PROJECT_ROOT
PROJECT_ROOT

## Required inputs

| Input | Description |
|---|---|
| `config/benchmark_config.yaml` | Benchmark config: `dataset_path`, `label_column`, `fractions`, `seeds`, `representations`, `k_neighbors`, `run_abundant_control`, `run_random_label_control` |
| `data/processed/pbmc5k_10x_citeseq_representations.h5ad` | Benchmark-ready AnnData with PCA embeddings in `obsm` (produced by `baseline_representations.ipynb`) |
| `results/metrics/rare_cell_benchmark_raw.csv` | Raw per-condition benchmark results (produced by `rare_cell_downsampling_benchmark.ipynb`) |

Run `rare_cell_downsampling_benchmark.ipynb` first to produce the raw metrics CSV required by `generate_summary_tables.py`.

## Run controls

Canonical command:
```bash
python scripts/run_controls.py --config config/benchmark_config.yaml
```

Config flags that control what is run:
- `run_abundant_control: true` — runs the benchmark on the most-abundant non-target cell type
- `run_random_label_control: true` — runs permuted-label evaluation at the smallest fraction
- `run_marker_removal_sensitivity: false` — optional sensitivity analysis (disabled by default)

Key functions used internally:
- `rarecell.controls.select_control_cell_type(labels, target, preferred)` — selects the control population
- `rarecell.controls.run_random_label_control(adata, ...)` — permuted-label benchmark
- `rarecell.benchmark.run_downsampling_benchmark(adata, ...)` — shared benchmark loop

In [ ]:
result = subprocess.run(
    [sys.executable, "scripts/run_controls.py", "--config", "config/benchmark_config.yaml"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## Generate summary tables

Canonical command:
```bash
python scripts/generate_summary_tables.py \
    --raw-csv results/metrics/rare_cell_benchmark_raw.csv \
    --control-csv results/metrics/abundant_cell_control_raw.csv
```

Key functions used internally:
- `rarecell.summaries.make_benchmark_summary(raw_df)` — long-format mean/std/n table
- `rarecell.summaries.make_best_method_summary(summary)` — best representation per (target, fraction, metric)
- `rarecell.summaries.make_abundant_control_summary(control_df)` — same schema for the control population

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/generate_summary_tables.py",
        "--raw-csv",
        "results/metrics/rare_cell_benchmark_raw.csv",
        "--control-csv",
        "results/metrics/abundant_cell_control_raw.csv",
    ],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## Expected outputs

| File | Script | Description |
|---|---|---|
| `results/metrics/abundant_cell_control_raw.csv` | `run_controls.py` | Raw per-condition metrics for the abundant-cell control |
| `results/tables/random_label_control.csv` | `run_controls.py` | Permuted-label control at the smallest fraction |
| `results/tables/benchmark_summary.csv` | `generate_summary_tables.py` | Long-format: mean, std, n_seeds per (target, representation, fraction, metric) |
| `results/tables/best_method_by_fraction.csv` | `generate_summary_tables.py` | Best and second-best representation per (target, fraction, metric) |
| `results/tables/abundant_cell_control_summary.csv` | `generate_summary_tables.py` | Same schema as benchmark_summary for the control cell type |

In [ ]:
outputs = [
    "results/metrics/abundant_cell_control_raw.csv",
    "results/tables/random_label_control.csv",
    "results/tables/benchmark_summary.csv",
    "results/tables/best_method_by_fraction.csv",
    "results/tables/abundant_cell_control_summary.csv",
]
[(path, (PROJECT_ROOT / path).exists()) for path in outputs]

## Summary

In [ ]:
print("Generated outputs:")
for path in outputs:
    p = PROJECT_ROOT / path
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {path}")
print()
print("Deviations from scripts:")
print("  - No file-level logging (scripts use setup logging to results/logs/).")
print("  - run_controls.py and generate_summary_tables.py are called sequentially here;")
print("    the scripts can be run independently in any order.")
print()
print("Note: results/metrics/rare_cell_benchmark_raw.csv must exist before running")
print("generate_summary_tables.py. Run rare_cell_downsampling_benchmark.ipynb first.")